# CDC COVID-19 Case Surveillance — Severity Analysis (Python/Pandas)

**Goal:** Analyze hospitalization, ICU admission, death outcomes, and severity trends over time using the CDC public-use case surveillance dataset.

**Dataset:** CDC Socrata dataset `vbim-akqf` (downloaded via API due to export limitations).  
**Local approach:** Reproducible 1,000,000-row sample saved as Parquet.

## Research Questions

1. Hospitalization rates by age group  
2. Death rates by underlying medical conditions  
3. ICU admission patterns by demographics  
4. Case severity trends over time (monthly)

## Data Fields Used

- cdc_report_dt (report date)
- age_group
- sex
- race_ethnicity_combined
- hosp_yn
- icu_yn
- death_yn
- medcond_yn
- current_status

## Data Acquisition

The dataset is very large (~106M rows). Browser export frequently fails, so data is retrieved through the Socrata API and stored locally as Parquet for performance.

In [ ]:
import requests, pandas as pd, time
from io import StringIO

BASE_URL = "https://data.cdc.gov/resource/vbim-akqf.csv"

select_columns = ",".join([
    "cdc_report_dt",
    "age_group",
    "sex",
    "race_ethnicity_combined",
    "hosp_yn",
    "icu_yn",
    "death_yn",
    "medcond_yn",
    "current_status"
])

TARGET_ROWS = 1_000_000
limit = 200_000
offset = 0
chunks = []

while offset < TARGET_ROWS:
    print(f"Downloading rows {offset} to {offset + limit}...")
    params = {
        "$select": select_columns, 
        "$limit": limit, 
        "$offset": offset}
    
    r = requests.get(BASE_URL, params=params)

    if r.status_code != 200:
        print("Error:", r.status_code, r.text[:300])
        break

    chunk = pd.read_csv(StringIO(r.text))
    
    if chunk.empty:
        break

    chunks.append(chunk)
    offset += limit
    time.sleep(1)

df = pd.concat(chunks, ignore_index=True)
df.to_parquet("C:/Users/gurba/Downloads/cdc-covid-project/data/processed/cdc_sample_1M.parquet", index=False)
print("Saved sample rows:", len(df))

In [ ]:
import pandas as pd

df = pd.read_parquet("C:/Users/gurba/Downloads/cdc-covid-project/data/processed/cdc_sample_1M.parquet")
df.shape

In [ ]:
df["age_group"].value_counts(dropna=False).head(20)
df["hosp_yn"].value_counts(dropna=False)
df["icu_yn"].value_counts(dropna=False)
df["death_yn"].value_counts(dropna=False)
df["medcond_yn"].value_counts(dropna=False)

In [ ]:
df.info()

In [ ]:
for col in df.columns:
    print("\n", col)
    print(df[col].unique()[:10])

In [ ]:
import pandas as pd

df_clean = df.copy()

# 1) Convert cdc_report_dt to datetime
df_clean["cdc_report_dt"] = pd.to_datetime(df_clean["cdc_report_dt"], errors="coerce")

# 2) Create month column (only works after datetime conversion)
df_clean["case_month"] = df_clean["cdc_report_dt"].dt.to_period("M").astype(str)

# 3) Keep only confirmed + probable cases (matches your dataset values exactly)
df_clean = df_clean[df_clean["current_status"].isin(["Laboratory-confirmed case", "Probable Case"])]

# 4) Create a known-hospitalization subset for the "known-only" denominator
df_hosp_known = df_clean[df_clean["hosp_yn"].isin(["Yes", "No"])]

df_clean[["cdc_report_dt", "case_month", "current_status"]].head()